# ResNet Binary Classification Pretrained Model

## Importing Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import json
import joblib
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import CIFAR100
from torchvision.utils import make_grid
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import random_split, ConcatDataset
from torchvision import transforms
from loguru import logger
import matplotlib.image as mpimg
import timm
from nazi_symbols_classification.training.data_preparation import get_image_paths, load_labels_df
from nazi_symbols_classification.training.torch_dataset_preparation import ImageData, get_device, to_device, ToDeviceLoader
from early_stopping_pytorch import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, roc_curve, auc, precision_recall_curve, average_precision_score


## Loading the Dataset

In [2]:
dir_name = os.path.dirname(os.getcwd())
dataset_path = f"{dir_name}/datasets/nazi-symbols-detection"

In [4]:
train_labels_df = load_labels_df(dataset_path, "train", binary=True)
valid_labels_df = load_labels_df(dataset_path, "val", binary=True)
test_labels_df = load_labels_df(dataset_path, "test", binary=True)

2025-06-10 16:20:55.547 | INFO     | __main__:load_labels_df:3 - Number of images in train is 75369
2025-06-10 16:20:55.582 | INFO     | __main__:load_labels_df:3 - Number of images in val is 15376
2025-06-10 16:20:55.600 | INFO     | __main__:load_labels_df:3 - Number of images in test is 15018


## Data Transformations and Dataloaders

In [5]:
stats = ((0.5074,0.4867,0.4411),(0.2011,0.1987,0.2025))
data_transf = transforms.Compose([transforms.ToPILImage(), 
                                  transforms.Grayscale(num_output_channels=3),
                                  transforms.Resize((224, 224)), 
                                  transforms.ToTensor(),
                                  transforms.Normalize(*stats)])

def create_datasets(batch_size, dataset_path=dataset_path):
    train_data = ImageData(df = train_labels_df,
                           data_directory = os.path.join(dataset_path, 'train'),
                           transform = data_transf,
                           label_column="contained_nazi")
    train_loader = DataLoader(dataset = train_data, batch_size = batch_size, shuffle=True)
    
    valid_data = ImageData(df = valid_labels_df, 
                           data_directory = os.path.join(dataset_path, 'val'), 
                           transform = data_transf,
                           label_column="contained_nazi")
    valid_loader = DataLoader(dataset = valid_data, batch_size = batch_size, shuffle=False)

    test_data = ImageData(df = test_labels_df, 
                           data_directory = os.path.join(dataset_path, 'test'), 
                           transform = data_transf,
                           label_column="contained_nazi")
    test_loader = DataLoader(dataset = test_data, batch_size = batch_size, shuffle=False)
    return train_loader, valid_loader, test_loader

In [7]:
batch_size = 16

train_loader, valid_loader, test_loader = create_datasets(batch_size, dataset_path)

In [9]:
device = get_device()
print(device)

train_dl = ToDeviceLoader(train_loader, device)
valid_dl = ToDeviceLoader(valid_loader, device)
test_dl = ToDeviceLoader(test_loader, device)

cuda


## Model Preparation

In [10]:
def accuracy(predicted, actual):
    _, predictions = torch.max(predicted, dim=1)
    return torch.tensor(torch.sum(predictions==actual).item()/len(predictions))

In [11]:
class BaseModel(nn.Module):
    def training_step(self,batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out,labels)
        return loss
    
    def validation_step(self,batch):
        images, labels = batch
        out = self(images)
        loss = F.cross_entropy(out,labels)
        acc = accuracy(out,labels)
        return {"val_loss":loss.detach(),"val_acc":acc}
    
    def validation_epoch_end(self,outputs):
        batch_losses = [loss["val_loss"] for loss in outputs]
        loss = torch.stack(batch_losses).mean()
        batch_accuracy = [accuracy["val_acc"] for accuracy in outputs]
        acc = torch.stack(batch_accuracy).mean()
        return {"val_loss":loss.item(),"val_acc":acc.item()}
    
    def epoch_end(self, epoch, result):
        print("Epoch [{}], last_lr: {:.5f}, train_loss: {:.4f}, val_loss: {:.4f}, val_acc: {:.4f}".format(
            epoch, result['lrs'][-1], result['train_loss'], result['val_loss'], result['val_acc']))

In [12]:
def conv_shortcut(in_channel, out_channel, stride):
    layers = [nn.Conv2d(in_channel, out_channel, kernel_size=(1,1), stride=(stride, stride)),
             nn.BatchNorm2d(out_channel)]
    return nn.Sequential(*layers)

def block(in_channel, out_channel, k_size,stride, conv=False):
    layers = None
    
    first_layers = [nn.Conv2d(in_channel,out_channel[0], kernel_size=(1,1),stride=(1,1)),
                    nn.BatchNorm2d(out_channel[0]),
                    nn.ReLU(inplace=True)]
    if conv:
        first_layers[0].stride=(stride,stride)
    
    second_layers = [nn.Conv2d(out_channel[0], out_channel[1], kernel_size=(k_size, k_size), stride=(1,1), padding=1),
                    nn.BatchNorm2d(out_channel[1])]

    layers = first_layers + second_layers
    
    return nn.Sequential(*layers)
    

class ResNet(BaseModel):
    
    def __init__(self, in_channels, num_classes, pretrained=False):
        super().__init__()
        self.model = None
        if pretrained:
            self.model = timm.create_model('resnet34', in_chans=in_channels, num_classes=num_classes, pretrained=True)
        else:
        
            self.stg1 = nn.Sequential(nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=(3),
                                                 stride=(1), padding=1),
                                       nn.BatchNorm2d(64),
                                       nn.ReLU(inplace=True),
                                       nn.MaxPool2d(kernel_size=3, stride=2))
            
            ##stage 2
            self.convShortcut2 = conv_shortcut(64,256,1)
            
            self.conv2 = block(64,[64,256],3,1,conv=True)
            self.ident2 = block(256,[64,256],3,1)
    
            
            ##stage 3
            self.convShortcut3 = conv_shortcut(256,512,2)
            
            self.conv3 = block(256,[128,512],3,2,conv=True)
            self.ident3 = block(512,[128,512],3,2)
    
            
            ##stage 4
            self.convShortcut4 = conv_shortcut(512,1024,2)
            
            self.conv4 = block(512,[256,1024],3,2,conv=True)
            self.ident4 = block(1024,[256,1024],3,2)
            
            
            ##Classify
            self.classifier = nn.Sequential(
                                           nn.AvgPool2d(kernel_size=(4)),
                                           nn.Flatten(),
                                           nn.Linear(1024, num_classes))
        
    def forward(self,inputs):
        if self.model:
            return self.model(inputs)
        out = self.stg1(inputs)
        
        #stage 2
        out = F.relu(self.conv2(out) + self.convShortcut2(out))
        out = F.relu(self.ident2(out) + out)
        out = F.relu(self.ident2(out) + out)
        out = F.relu(self.ident2(out) + out)
        
        #stage3
        out = F.relu(self.conv3(out) + (self.convShortcut3(out)))
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        
        #stage4             
        out = F.relu(self.conv4(out) + (self.convShortcut4(out)))
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        
        #Classify
        out = self.classifier(out)#100x1024
        
        return out
        

In [13]:
print("Available Vision Transformer Models: ")
timm.list_models("*resnet*", pretrained=True)

Available Vision Transformer Models: 


['cspresnet50.ra_in1k',
 'eca_resnet33ts.ra2_in1k',
 'ecaresnet26t.ra2_in1k',
 'ecaresnet50d.miil_in1k',
 'ecaresnet50d_pruned.miil_in1k',
 'ecaresnet50t.a1_in1k',
 'ecaresnet50t.a2_in1k',
 'ecaresnet50t.a3_in1k',
 'ecaresnet50t.ra2_in1k',
 'ecaresnet101d.miil_in1k',
 'ecaresnet101d_pruned.miil_in1k',
 'ecaresnet269d.ra2_in1k',
 'ecaresnetlight.miil_in1k',
 'gcresnet33ts.ra2_in1k',
 'gcresnet50t.ra2_in1k',
 'inception_resnet_v2.tf_ens_adv_in1k',
 'inception_resnet_v2.tf_in1k',
 'lambda_resnet26rpt_256.c1_in1k',
 'lambda_resnet26t.c1_in1k',
 'lambda_resnet50ts.a1h_in1k',
 'legacy_seresnet18.in1k',
 'legacy_seresnet34.in1k',
 'legacy_seresnet50.in1k',
 'legacy_seresnet101.in1k',
 'legacy_seresnet152.in1k',
 'nf_resnet50.ra2_in1k',
 'resnet10t.c3_in1k',
 'resnet14t.c3_in1k',
 'resnet18.a1_in1k',
 'resnet18.a2_in1k',
 'resnet18.a3_in1k',
 'resnet18.fb_ssl_yfcc100m_ft_in1k',
 'resnet18.fb_swsl_ig1b_ft_in1k',
 'resnet18.gluon_in1k',
 'resnet18.tv_in1k',
 'resnet18d.ra2_in1k',
 'resnet18d.ra4

In [14]:
model = ResNet(3,2, pretrained=True)

In [15]:
model = to_device(model, device)

In [16]:
@torch.no_grad()
def evaluate(model,valid_dl):
    model.eval()
    outputs = [model.validation_step(batch) for batch in valid_dl]
    return model.validation_epoch_end(outputs)

In [17]:
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def fit (epochs, train_dl, valid_dl, model, optimizer, max_lr, weight_decay, scheduler, grad_clip=None, patience=10):
    torch.cuda.empty_cache()
    
    history = []
    
    optimizer = optimizer(model.parameters(), max_lr, weight_decay = weight_decay)
    
    scheduler = scheduler(optimizer, max_lr, epochs=epochs, steps_per_epoch=len(train_dl))

    early_stopping = EarlyStopping(patience=patience, verbose=True, path="resnet-output/resnet-binary-pretrained-size-224-batch-16.pt")
    
    for epoch in range(epochs):
        model.train()
        
        train_loss = []
        
        lrs = []
        
        for batch in train_dl:
            loss = model.training_step(batch)
            
            train_loss.append(loss)
            
            loss.backward()
            
            if grad_clip:
                nn.utils.clip_grad_value_(model.parameters(), grad_clip)
            
            optimizer.step()
            optimizer.zero_grad()
            
            scheduler.step()
            lrs.append(get_lr(optimizer))
        result = evaluate(model, valid_dl)
        result["train_loss"] = torch.stack(train_loss).mean().item()
        result["lrs"] = lrs
        
        model.epoch_end(epoch,result)
        history.append(result)

        early_stopping(result["val_loss"], model)
        
        if early_stopping.early_stop:
            print("Early stopping")
            break
        
    return history

## Training the Model

In [18]:
epochs = 100
optimizer = torch.optim.Adam
max_lr = 1e-3
grad_clip = 0.1
weight_decay = 1e-5
scheduler = torch.optim.lr_scheduler.OneCycleLR
patience = 10

In [ ]:
%%time
history = fit(epochs=epochs, train_dl=train_dl, valid_dl=valid_dl, model=model, 
              optimizer=optimizer, max_lr=max_lr, grad_clip=grad_clip, patience=patience,
              weight_decay=weight_decay, scheduler=torch.optim.lr_scheduler.OneCycleLR)

Epoch [0], last_lr: 0.00004, train_loss: 0.0868, val_loss: 0.0210, val_acc: 0.9930
Validation loss decreased (inf --> 0.020985).  Saving model ...
Epoch [1], last_lr: 0.00005, train_loss: 0.0307, val_loss: 0.0181, val_acc: 0.9934
Validation loss decreased (0.020985 --> 0.018117).  Saving model ...


In [37]:
with open('resnet-binary-pretrained-history.json', 'w') as f:
    f.write(json.dumps(history))

## Evaluating the Model

In [43]:
model.load_state_dict(torch.load("resnet-output/resnet-binary-pretrained-size-224-batch-16.pt", weights_only=True))
model.eval()

ResNet(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (drop_block): Identity()
        (act1): ReLU(inplace=True)
        (aa): Identity()
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act2): ReLU(inplace=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), 

In [44]:
outputs = []
y_true = []
probabilities = []
out = None
for batch in ToDeviceLoader(test_loader, "cuda"):
    images, labels = batch
    with torch.no_grad():
        out = model(images)
    probs = torch.nn.functional.softmax(out, dim=1)
    conf, classes = torch.max(probs, 1)
    # print([le.classes_[predicted[i]] for i in range(len(images))])
    y_true += labels.cpu()
    outputs += classes.cpu()
    probabilities += probs.cpu()

In [45]:
probs = []
for prob in probabilities:
    probs.append(prob.detach().numpy()[1])
probs[:3]

[4.662742e-05, 0.22121754, 6.101811e-06]

In [47]:
predict_result = dict(y_true=y_true, probs=probs,outputs=outputs)

joblib.dump(predict_result, 'resnet-output/resnet-binary-pretrained-predicted-results.joblib')

['resnet-binary-pretrained-predicted-results.joblib']

## Model Evaluation Metrics

In [48]:
fpr, tpr, roc_threshold = roc_curve(y_true, probs)
roc_auc = auc(fpr, tpr)
precision, recall, pr_threshold = precision_recall_curve(y_true, probs)
average_precision = average_precision_score(y_true, probs)
roc_auc_result = dict(precision=precision, recall=recall, pr_threshold=pr_threshold, average_precision=average_precision,
                      fpr=fpr, tpr=tpr, roc_threshold=roc_threshold, roc_auc=roc_auc)
roc_auc_result = dict(fpr=fpr, tpr=tpr, threshold=roc_threshold, roc_auc=roc_auc, 
                      precision=precision, recall=recall, pr_threshold=pr_threshold, average_precision=average_precision)

print(classification_report(y_true, outputs, digits=3))
print("accuracy score:", accuracy_score(y_true, outputs))
print("roc auc score:", roc_auc_score(y_true, outputs))

              precision    recall  f1-score   support

           0      0.999     0.998     0.998     14813
           1      0.843     0.893     0.867       205

    accuracy                          0.996     15018
   macro avg      0.921     0.945     0.933     15018
weighted avg      0.996     0.996     0.996     15018

accuracy score: 0.9962711412971101
roc auc score: 0.9451938228286624


In [25]:
joblib.dump(roc_auc_result, 'resnet-output/resnet-binary-pretrained-roc-auc-results.joblib')

['resnet-binary-pretrained-roc-auc-results.joblib']

In [33]:
import gc

# del model
gc.collect()
torch.cuda.empty_cache()

['resnet-binary-pretrained-roc-auc-results.joblib']